In [ ]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/FRAMES'

import os
os.makedirs(os.path.join(DATA_DIR, 'predictions'), exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install transformers datasets rank_bm25 torch tqdm accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.3 MB/s eta 0:00:00


In [ ]:
import json, pickle, os, re, torch, ast
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load FRAMES
frames = load_dataset('google/frames-benchmark')['test']

# Load BM25 index and corpus
with open(os.path.join(DATA_DIR, 'corpus.json'), encoding='utf-8') as f:
    corpus = json.load(f)
with open(os.path.join(DATA_DIR, 'titles.json'), encoding='utf-8') as f:
    titles = json.load(f)
with open(os.path.join(DATA_DIR, 'bm25_index.pkl'), 'rb') as f:
    bm25 = pickle.load(f)

print(f'FRAMES: {len(frames)} questions, Corpus: {len(corpus)} articles')

# Only answering 50% of the questions to save runtime
EVAL_SUBSET = 0.5


if EVAL_SUBSET < 1.0:
    import random
    from collections import defaultdict
    random.seed(42)
    type_indices = defaultdict(list)
    for i, row in enumerate(frames):
        key = tuple(sorted(row.get('reasoning_types') or ['Unknown']))
        type_indices[key].append(i)
    subset_indices = set()
    for key, indices in type_indices.items():
        k = max(1, round(len(indices) * EVAL_SUBSET))
        subset_indices.update(random.sample(indices, k))
    frames = [frames[i] for i in sorted(subset_indices)]
    print(f'Subset: {len(frames)} questions ({EVAL_SUBSET:.0%} stratified, seed=42)')

def get_wiki_links(row):
    links = row['wiki_links']
    if isinstance(links, str):
        try:
            links = ast.literal_eval(links)
        except:
            links = [links]
    return [l for l in links if isinstance(l, str) and len(l) > 3]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

test.tsv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/824 [00:00<?, ? examples/s]

FRAMES: 824 questions, Corpus: 3963 articles
Subset: 412 questions (50% stratified, seed=42)


In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-14B-Instruct'
print(f'Loading {MODEL_ID}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()

Loading Qwen/Qwen2.5-14B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [ ]:
def llm_generate(prompt, max_new_tokens=256):
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def retrieve_bm25(query, k=3):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = scores.argsort()[-k:][::-1]
    return [(titles[i], corpus[titles[i]]) for i in top_idx]

def save_predictions(name, preds):
    path = os.path.join(DATA_DIR, 'predictions', f'{name}.json')
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)
    print(f'Saved {len(preds)} predictions -> {path}')

In [ ]:
# Prompt templates
# Prompts made using ChatGPT
QUERY_GEN_PROMPT = """You are answering a complex question by searching Wikipedia one step at a time.

Question: {question}

Information retrieved so far:
{context}

Write ONE short search query (5-10 words) to find the next piece of information needed.
Rules:
- On the first step, identify the first specific entity or fact to look up
- Each query must target ONE specific entity, person, event, or date
- Never paste the full question as a query — break it into atomic lookups
- Do not repeat a previous query

Previous queries: {prev_queries}

Next search query (output the query only, no explanation, no labels):"""

ANSWER_PROMPT = """Answer the question using the Wikipedia articles below as your primary source.
Read the articles carefully. If the answer is in the articles, use it.
If the articles do not contain enough information, use your best knowledge but keep the answer short.

{context}

Question: {question}

Answer (be concise — one sentence or a short phrase):"""

def get_top_passages(content, query, n=3, passage_chars=500, stride=250):
    """Split article into overlapping passages; return top-n by query word overlap.

    Replaces get_relevant_section: instead of picking one section by header match,
    we slide a window over the full article so buried facts are reachable.
    """
    passages = []
    for start in range(0, len(content), stride):
        chunk = content[start:start + passage_chars]
        if len(chunk) < 50:
            continue
        passages.append(chunk)
    if not passages:
        return content[:passage_chars]
    query_words = set(query.lower().split())
    scored = sorted(passages, key=lambda p: -len(query_words & set(p.lower().split())))
    return ' [...] '.join(scored[:n])

def format_context(retrieved_articles, query='', n_passages=3, passage_chars=500):
    """Format accumulated articles as context, using top-n relevant passages per article."""
    parts = []
    for title, content in retrieved_articles:
        passages = get_top_passages(content, query, n=n_passages, passage_chars=passage_chars)
        parts.append(f'[{title}]\n{passages}')
    return '\n\n'.join(parts)

def multihop_rag(question, num_hops=3, k_per_hop=5):
    """
    Iterative retrieval loop.
    Returns dict with answer, retrieved_titles, queries_used.
    """
    retrieved_articles = []
    retrieved_titles_set = set()
    queries_used = []

    for hop in range(num_hops):
        if hop == 0:
            query = question
        else:
            context_str = format_context(retrieved_articles, query=question, n_passages=2, passage_chars=300)
            prompt = QUERY_GEN_PROMPT.format(
                question=question,
                context=context_str[:1500],
                prev_queries=', '.join(f'"{q}"' for q in queries_used) or 'None'
            )
            query = llm_generate(prompt, max_new_tokens=64).strip().split('\n')[0].strip()
        queries_used.append(query)

        results = retrieve_bm25(query, k=k_per_hop)
        for title, content in results:
            if title not in retrieved_titles_set:
                retrieved_articles.append((title, content))
                retrieved_titles_set.add(title)

    context_str = format_context(retrieved_articles, query=question, n_passages=3, passage_chars=700)
    answer_prompt = ANSWER_PROMPT.format(context=context_str, question=question)
    answer = llm_generate(answer_prompt, max_new_tokens=512)

    return {
        'answer': answer,
        'retrieved_titles': list(retrieved_titles_set),
        'queries_used': queries_used,
        'num_hops': num_hops
    }

# Check one example
print('Testing multi-hop on example...')
test = multihop_rag(frames[0]['Prompt'], num_hops=3)
print('Question:', frames[0]['Prompt'][:80])
print('Gold:', frames[0]['Answer'])
print('Predicted:', test['answer'][:200])
print('Queries used:', test['queries_used'])
print('Retrieved:', test['retrieved_titles'])

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Testing multi-hop on example...
Question: How many years earlier would Punxsutawney Phil have to be canonically alive to h
Gold: 87
Predicted: To have made a prediction in the same state as the US Capitol (Washington D.C.), Punxsutawney Phil would need to have been canonically alive at least since Pennsylvania was a state (1787), which is ab
Queries used: ['How many years earlier would Punxsutawney Phil have to be canonically alive to have made a Groundhog Day prediction in the same state as the US capitol?', 'When was Punxsutawney Phil first mentioned as making a prediction?', 'When was the US capital established?']
Retrieved: ['The Hangover', 'List of capitals in the United States', 'Aristotle', 'Punxsutawney Phil', 'Margaret Truman', 'Capital One Arena', 'Nuapada district', 'Prague', 'Nike, Inc.', 'Kenny McCormick', 'Linus Pauling', 'List of Liverpool F.C. managers', 'The Hangover Part II']


In [ ]:
# Run 1 hop RAG
NUM_HOPS_SINGLE = 1
K_PER_HOP = 5

rag_single_preds = []
for row in tqdm(frames, desc='C1 Single-step RAG'):
    result = multihop_rag(row['Prompt'], num_hops=NUM_HOPS_SINGLE, k_per_hop=K_PER_HOP)
    rag_single_preds.append({
        'id': row.get('id', ''),
        'question': row['Prompt'],
        'gold_answer': row['Answer'],
        'predicted': result['answer'],
        'retrieved_titles': result['retrieved_titles'],
        'queries_used': result['queries_used'],
        'num_hops': NUM_HOPS_SINGLE,
        'reasoning_types': row['reasoning_types'],
        'n_gold_articles': len(get_wiki_links(row)),
        'gold_wiki_links': get_wiki_links(row)
    })

save_predictions('c1_rag_single_hop', rag_single_preds)

C1 Single-step RAG:   0%|          | 0/412 [00:00<?, ?it/s]

Saved 412 predictions -> /content/drive/MyDrive/FRAMES/predictions/c1_rag_single_hop.json


In [ ]:
# Run multi-hop RAG
NUM_HOPS_MULTI = 3
K_PER_HOP = 5

rag_multi_preds = []
for row in tqdm(frames, desc='C2 Multi-hop RAG (3 hops)'):
    result = multihop_rag(row['Prompt'], num_hops=NUM_HOPS_MULTI, k_per_hop=K_PER_HOP)
    rag_multi_preds.append({
        'id': row.get('id', ''),
        'question': row['Prompt'],
        'gold_answer': row['Answer'],
        'predicted': result['answer'],
        'retrieved_titles': result['retrieved_titles'],
        'queries_used': result['queries_used'],
        'num_hops': NUM_HOPS_MULTI,
        'reasoning_types': row['reasoning_types'],
        'n_gold_articles': len(get_wiki_links(row)),
        'gold_wiki_links': get_wiki_links(row)
    })

save_predictions('c2_rag_multihop_3', rag_multi_preds)

C2 Multi-hop RAG (3 hops):   0%|          | 0/412 [00:00<?, ?it/s]

Saved 412 predictions -> /content/drive/MyDrive/FRAMES/predictions/c2_rag_multihop_3.json


In [ ]:
# Could test 5-hops too, depending on time. Just need to change False to True and it will run.
# Would in that case be saved as C3.

RUN_5_HOP = False

if RUN_5_HOP:
    rag_5hop_preds = []
    for row in tqdm(frames, desc='C3 Multi-hop RAG (5 hops)'):
        result = multihop_rag(row['Prompt'], num_hops=5, k_per_hop=3)
        rag_5hop_preds.append({
            'id': row.get('id', ''),
            'question': row['Prompt'],
            'gold_answer': row['Answer'],
            'predicted': result['answer'],
            'retrieved_titles': result['retrieved_titles'],
            'queries_used': result['queries_used'],
            'num_hops': 5,
            'reasoning_types': row['reasoning_types'],
            'n_gold_articles': len(get_wiki_links(row)),
            'gold_wiki_links': get_wiki_links(row)
        })
    save_predictions('c3_rag_multihop_5', rag_5hop_preds)

In [ ]:
# Checking recall

import re

def title_from_url(url):
    match = re.search(r'/wiki/(.+)', url)
    return match.group(1).replace('_', ' ') if match else url

def retrieval_recall(preds):
    recalls = []
    for p in preds:
        gold = set(title_from_url(u).lower() for u in p['gold_wiki_links'])
        retrieved = set(t.lower() for t in p['retrieved_titles'])
        if gold:
            recalls.append(len(gold & retrieved) / len(gold))
    return sum(recalls) / len(recalls) if recalls else 0.0

r1 = retrieval_recall(rag_single_preds)
r3 = retrieval_recall(rag_multi_preds)

print(f'Recall (fraction of gold articles retrieved):')
print(f'  Single-hop RAG (1 hop, k=5): {r1:.3f}')
print(f'  Multi-hop RAG  (3 hops, k=5): {r3:.3f}')
print(f'\nExpected: multi-hop > single-hop')

Recall (fraction of gold articles retrieved):
  Single-hop RAG (1 hop, k=5): 0.437
  Multi-hop RAG  (3 hops, k=5): 0.616

Expected: multi-hop > single-hop
Done. Run notebook 04 for LLM-as-judge evaluation.
